In [37]:
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver #Used for saving messages temprarily to the RAM
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END, START
from typing import Annotated, TypedDict, Literal

In [38]:
OLLAMA_MODEL = "qwen2.5-coder:14b"
OLLAMA_BASE_URL = "http://localhost:11434"
model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

In [39]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [40]:
def chat(state:ChatState):
    messages=state['messages']
    response=model.invoke(messages).content
    return {'messages': [AIMessage(content=response)]}

In [41]:
checkpointer=MemorySaver() #used in saving the previous messages to RAM
graph= StateGraph(ChatState)

graph.add_node("chat",chat)

graph.add_edge(START,"chat")
graph.add_edge("chat",END)

chatbot=graph.compile(checkpointer=checkpointer) # checkpointer=checkpointer used in saving the previous messages to RAM

In [42]:
thread_id="1"   # used in saving the previous messages to RAM

while True:
    user_message= input("Type ypu message here: ")
    if user_message.strip().lower() in ['quit',"bye"]:
        break
    print(user_message)

    response = chatbot.invoke(
    {"messages": [HumanMessage(content=user_message)]},
    config={"configurable": {"thread_id": thread_id}}       #used in saving the previous messages to RAM
    )

    print(response["messages"][-1].content)

hello i am bilal
Hello Bilal! How can I assist you today?
what is my name
Your name is Bilal.
